In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [38]:
# Carrega o relatório de vendas do Mercado Livre
# header=5 pula as 6 linhas de "cabeçalho visual" do relatório
# (texto explicativo, link, título com data, categorias agrupadas)
df = pd.read_csv("../data/raw/dados-meli-12-setembro.csv", header=5)

print(f"Shape: {df.shape[0]} vendas x {df.shape[1]} colunas")
df.head()

Shape: 181 vendas x 64 colunas


,N.º de venda,Data da venda,Estado,Descrição do status,Pacote de diversos produtos,Pertence a um kit,Unidades,Receita por produtos (BRL),Receita por acréscimo no preço (pago pelo comprador),Taxa de parcelamento equivalente ao acréscimo,...,Revisado pelo Mercado Livre,Data de revisão,Dinheiro liberado,Resultado,Destino,Motivo do resultado,Unidades.2,Reclamação aberta,Reclamação encerrada,Em mediação
0,2000014993151901,12 de setembro de 2026 00:25 hs.,Cancelada pelo comprador,Cancelou e especificou outro problema.,Sim,Não,1.0,199.50,25.66,-25.66,...,,,,,,,NaN,Não,NaN,Não
1,2000014985711089,11 de setembro de 2026 14:59 hs.,A caminho,Chega entre os dias 23 e 28 de setembro,Sim,Não,1.0,199.80,NaN,NaN,...,,,,,,,NaN,Não,NaN,Não
2,2000014984624173,11 de setembro de 2026 13:55 hs.,Entregue,Chegou em 14 de setembro,Sim,Não,1.0,149.94,NaN,NaN,...,,,,,,,NaN,Não,NaN,Não
3,2000014984400301,11 de setembro de 2026 13:37 hs.,Entregue,Chegou em 14 de setembro,Sim,Não,1.0,149.94,NaN,NaN,...,,,,,,,NaN,Não,NaN,Não
4,2000018401478462,10 de setembro de 2026 23:55 hs.,A caminho,Chegará hoje,Não,Não,1.0,135.18,NaN,NaN,...,,,,,,,NaN,Não,NaN,Não


In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 181 entries, 0 to 180
Data columns (total 64 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   N.º de venda                                           181 non-null    int64  
 1   Data da venda                                          181 non-null    str    
 2   Estado                                                 181 non-null    str    
 3   Descrição do status                                    181 non-null    str    
 4   Pacote de diversos produtos                            181 non-null    str    
 5   Pertence a um kit                                      181 non-null    str    
 6   Unidades                                               174 non-null    float64
 7   Receita por produtos (BRL)                             166 non-null    float64
 8   Receita por acréscimo no preço (pago pelo comprador)   44 non

In [40]:
df.isna().sum().sort_values(ascending=False)

Custo de envio com base nas medidas e peso declarados    181
Custo por diferenças nas medidas e no peso do pacote     181
Unidades.2                                               180
Reclamação encerrada                                     177
Cancelamentos e reembolsos (BRL)                         166
                                                        ... 
Resultado                                                  0
Motivo do resultado                                        0
Destino                                                    0
Reclamação aberta                                          0
Em mediação                                                0
Length: 64, dtype: int64

In [41]:
# Conta quantas vendas existem em cada status.
# Ajuda a entender o desbalanceamento das classes: a maioria costuma ser
# "Entregue", enquanto cancelamentos/devoluções são minoria — isso importa
# na hora de escolher a métrica de avaliação do modelo mais pra frente.
df["Estado"].value_counts()

Estado
Entregue                                               154
Cancelada pelo comprador                                 6
Pacote de 2 produtos                                     6
Devolução finalizada com reembolso para o comprador      5
A caminho                                                4
Devolução a caminho                                      1
Devolução atrasada                                       1
Pacote de 3 produtos                                     1
Devolvido no dia 26 de agosto                            1
Venda entregue                                           1
Devolvido no dia 14 de agosto                            1
Name: count, dtype: int64

In [42]:
# Mostra o tipo de dado de cada coluna (object, int64, float64, etc.).
# Serve pra identificar o que precisa ser convertido na limpeza:
# datas que ainda estão como texto, valores em BRL como string, etc.
df.dtypes

N.º de venda                     int64
Data da venda                      str
Estado                             str
Descrição do status                str
Pacote de diversos produtos        str
                                ...   
Motivo do resultado                str
Unidades.2                     float64
Reclamação aberta                  str
Reclamação encerrada           float64
Em mediação                        str
Length: 64, dtype: object

In [43]:
# Conta quantos SKUs e quantos títulos de anúncio distintos existem.
# Importante porque cada linha do dataset é uma VENDA, mas o objetivo do
# projeto é prever por ITEM/produto — então depois vai ser preciso agrupar
# as vendas por SKU (ou por título de anúncio, se o SKU tiver muito lixo).
df["SKU"].nunique(), df["Título do anúncio"].nunique()

(66, 50)

In [44]:
# O relatório do Mercado Livre repete nomes de coluna em seções diferentes
# (ex: "Unidades" aparece em Vendas, Devoluções e Reclamações; "Estado" aparece
# em Vendas — status da venda — e em Compradores — estado/UF do comprador).
# O pandas resolve isso sozinho com sufixo .1, .2, mas o nome fica ambíguo.
# Aqui a gente renomeia usando o grupo de origem (visto no relatório original)
# pra deixar claro o que cada coluna significa.
renomear = {
    "Estado.1": "uf_comprador",
    "Endereço.1": "endereco_comprador",
    "Unidades.1": "unidades_devolvidas",
    "Unidades.2": "unidades_reclamadas",
    "Forma de entrega.1": "forma_entrega_devolucao",
    "Data a caminho.1": "data_a_caminho_devolucao",
    "Data de entrega.1": "data_entrega_devolucao",
    "Transportador.1": "transportador_devolucao",
    "Número de rastreamento.1": "rastreamento_devolucao",
    "URL de acompanhamento.1": "url_acompanhamento_devolucao",
}
df = df.rename(columns=renomear)

In [45]:
# As datas vêm por extenso em português ("12 de setembro de 2026 00:25 hs."),
# formato que o pandas não reconhece sozinho. Criamos um dicionário de meses
# e convertemos manualmente pra datetime de verdade — isso permite depois
# extrair dia da semana, mês, etc. como features pro modelo.
meses = {
    "janeiro": "01", "fevereiro": "02", "março": "03", "abril": "04",
    "maio": "05", "junho": "06", "julho": "07", "agosto": "08",
    "setembro": "09", "outubro": "10", "novembro": "11", "dezembro": "12",
}

def parse_data_ptbr(texto):
    # Ignora valores nulos (algumas colunas de data podem estar vazias)
    if pd.isna(texto):
        return pd.NaT
    for nome_mes, numero_mes in meses.items():
        texto = texto.replace(nome_mes, numero_mes)
    # Remove "de" e "hs." pra sobrar só números e ':'
    texto = texto.replace(" de ", "/").replace(" hs.", "")
    return pd.to_datetime(texto, format="%d/%m/%Y %H:%M")

df["Data da venda"] = df["Data da venda"].apply(parse_data_ptbr)

In [46]:
# Define a coluna-alvo (target) a partir do status da venda.
# 1 = venda efetivada (entregue), 0 = não efetivada (cancelada/devolvida).
# Isso vira a base pro que o modelo (Árvore de Decisão / Random Forest) vai prever.
status_efetivado = ["Entregue", "Venda entregue"]
df["venda_efetivada"] = df["Estado"].isin(status_efetivado).astype(int)

df["venda_efetivada"].value_counts()

venda_efetivada
1    155
0     26
Name: count, dtype: int64